# SONAR Quick Start

This notebook demonstrates how to load the SONAR dataset, explore its properties,
run a graph anomaly detector, and evaluate the results.

## Installation

```bash
pip install "sonar-graph[benchmark,notebooks]"
```

## 1. Load the SONAR Small Dataset

In [ ]:
from sonar import SONAR, dataset_summary

# Load homogeneous graph with anomalies (auto-downloaded)
dataset = SONAR(root="./data", name="small", anomalies=True)
data = dataset[0]
print(dataset)
print(f"Nodes: {data.num_nodes}, Edges: {data.num_edges}")
print(f"Features: {data.x.shape[1]}, Anomalies: {int(data.y_outlier.sum())}")

In [ ]:
# Load heterogeneous (clean) variant
dataset_hetero = SONAR(root="./data", name="small", anomalies=False,
                       representation="heterogeneous")
data_hetero = dataset_hetero[0]
print(dataset_hetero)
print(data_hetero)

## 2. Dataset Summary

In [ ]:
# Homogeneous summary
summary = dataset_summary(data)
for k, v in summary.items():
    print(f"{k}: {v}")

In [ ]:
# Heterogeneous summary
summary_hetero = dataset_summary(data_hetero)
for k, v in summary_hetero.items():
    print(f"{k}: {v}")

## 3. Visualize Graph Properties

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch_geometric.utils import degree

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Degree distribution
deg = degree(data.edge_index[0], num_nodes=data.num_nodes)
axes[0].hist(deg.numpy(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count")
axes[0].set_title("Degree Distribution")
axes[0].set_yscale("log")

# Feature distribution (first feature)
axes[1].hist(data.x[:, 0].numpy(), bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Feature 0 value")
axes[1].set_ylabel("Count")
axes[1].set_title("Feature 0 Distribution")

# Anomaly label distribution
labels, counts = torch.unique(data.y_outlier, return_counts=True)
axes[2].bar(["Normal", "Anomaly"], counts.numpy(), color=["steelblue", "coral"],
            edgecolor="black")
axes[2].set_ylabel("Count")
axes[2].set_title("Label Distribution")

plt.tight_layout()
plt.show()

## 4. Run a Detector (DOMINANT)

In [ ]:
from pygod.detector import DOMINANT

detector = DOMINANT(epoch=5, gpu=-1, verbose=1)
detector.fit(data)
pred, score = detector.predict(data, return_pred=True, return_score=True)

print(f"Outliers detected: {int(pred.sum())}")
print(f"Score range: [{score.min():.4f}, {score.max():.4f}]")

## 5. Evaluate

In [ ]:
from sonar import evaluate_detector

metrics = evaluate_detector(data.y_outlier, score)
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

## 6. Compare Multiple Detectors

In [ ]:
import pandas as pd
from pygod.detector import DOMINANT, ANOMALOUS
from sonar import evaluate_detector

detectors = {
    "DOMINANT": DOMINANT(epoch=5, gpu=-1),
    "ANOMALOUS": ANOMALOUS(gpu=-1),
}

results = []
for name, det in detectors.items():
    print(f"Running {name}...")
    det.fit(data)
    _, score = det.predict(data, return_pred=True, return_score=True)
    metrics = evaluate_detector(data.y_outlier, score)
    metrics["detector"] = name
    results.append(metrics)

df = pd.DataFrame(results).set_index("detector")
df